### Replace the citation numbers in a saved Perplexity dialogue with matching literature note wikilinks.

In [1]:
%load_ext autoreload
%autoreload 2

from icecream import ic
import pathlib as pl

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
import sys
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw

tmp_dir = rfw.refwrangle_test_dir / 'tmp'

perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
obsidian_citekeys_file = rfw.refwrangle_test_dir / "dat" / 'obsnotecitekeys.csv'

output_file = tmp_dir / "tmp_perplex_dialog_with_files.md"

In [2]:
import re
import pandas as pd
from urllib.parse import urlparse, urlunparse

def replace_citations(markdown_file, csv_file, output_file):
    # Read the CSV file and create a dictionary of normalized URL to citekey mappings
    df = pd.read_csv(csv_file)
    url_to_citekey = {rfw.normalize_url(url): citekey for url, citekey in zip(df.url, df.citekey)}

    # Read the Markdown file
    with open(markdown_file, 'r') as mdfile:
        content = mdfile.read()

    # Split the content into body and citations
    parts = content.split("\nCitations:\n")
    if len(parts) != 2:
        raise Exception("Couldn't find Citations section")
    
    body, citations = parts

    # Extract citations and their corresponding normalized URLs
    citation_urls = re.findall(r'\[(\d+)\]\s+(https?://\S+)', citations)
    url_to_number = {rfw.normalize_url(url): num for num, url in citation_urls}

    # Replace citations in the body text with wikilinks
    def replace_citation(match):
        num = match.group(1)
        normalized_url = next((url for url, cite_num in url_to_number.items() if cite_num == num), None)
        if normalized_url and normalized_url in url_to_citekey:
            return f'[[{url_to_citekey[normalized_url]}]]'
        return match.group(0)

    body = re.sub(r'\[(\d+)\]', replace_citation, body)

    # Write the modified content to the output file
    with open(output_file, 'w') as outfile:
        outfile.write(body)

In [ ]:
ic(perplexity_dialog_file, obsidian_citekeys_file, output_file)

replace_citations(perplexity_dialog_file, obsidian_citekeys_file, output_file)